# K-fold of XGB

In [ ]:
import pandas as pd
import numpy as np
import os

from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import KFold
from xgboost import XGBRegressor

In [13]:
data_path = r"path\Train_Test_Verification_Data.xlsx"

data = pd.read_excel(data_path,sheet_name="100 divisions_K-folds")

data_features = data.iloc[:, :10]
data_label = data.iloc[:, -1]

X_all = data_features.to_numpy(dtype=np.float32)
y_all = data_label.to_numpy(dtype=np.float32).ravel()

print("All data shape:", data.shape)
print("Feature shape:", X_all.shape)
print("Label shape:", y_all.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'path\\Train_Test_Verification_Data.xlsx'

In [11]:
def evaluate_xgb(model, X_eval, y_eval):
    y_pred = model.predict(X_eval)

    r2 = r2_score(y_eval, y_pred)
    rmse = np.sqrt(mean_squared_error(y_eval, y_pred))

    return r2, rmse



N_EXPERIMENTS = 178
POINTS_PER_EXPERIMENT = 574

expected_rows = N_EXPERIMENTS * POINTS_PER_EXPERIMENT
actual_rows = len(data)

print("Expected rows:", expected_rows)
print("Actual rows:", actual_rows)



experiment_ids = np.repeat(
    np.arange(N_EXPERIMENTS),
    POINTS_PER_EXPERIMENT
)

data_with_id = data.copy()
data_with_id["Experiment_ID"] = experiment_ids

all_experiment_ids = np.arange(N_EXPERIMENTS)


XGB_N_ESTIMATORS = 50
XGB_MAX_DEPTH = 6

K_LIST = [5, 6, 7, 8, 9, 10]

Expected rows: 102172
Actual rows: 102172


In [12]:
fold_results = []

for K in K_LIST:
    print("\n" + "=" * 70)
    print(f"XGB Group-level {K}-fold cross-validation")
    print("=" * 70)

    kf = KFold(
        n_splits=K,
        shuffle=True,
        random_state=42
    )

    for fold_idx, (train_exp_index, val_exp_index) in enumerate(
        kf.split(all_experiment_ids),
        start=1
    ):
        train_exp_ids = all_experiment_ids[train_exp_index]
        val_exp_ids = all_experiment_ids[val_exp_index]

        train_mask = data_with_id["Experiment_ID"].isin(train_exp_ids).to_numpy()
        val_mask = data_with_id["Experiment_ID"].isin(val_exp_ids).to_numpy()

        feature_train = X_all[train_mask]
        label_train = y_all[train_mask]

        feature_val = X_all[val_mask]
        label_val = y_all[val_mask]

        model = XGBRegressor(
            n_estimators=XGB_N_ESTIMATORS,
            max_depth=XGB_MAX_DEPTH,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )

        model.fit(feature_train, label_train)

        train_r2, train_rmse = evaluate_xgb(
            model,
            feature_train,
            label_train
        )

        val_r2, val_rmse = evaluate_xgb(
            model,
            feature_val,
            label_val
        )

        fold_results.append({
            "Model": "XGB",
            "K": K,
            "Fold": fold_idx,

            "Train_experiment_number": len(train_exp_ids),
            "Val_experiment_number": len(val_exp_ids),

            "Train_data_points": len(label_train),
            "Val_data_points": len(label_val),

            "Train_R2": train_r2,
            "Val_R2": val_r2,
            "Train_RMSE": train_rmse,
            "Val_RMSE": val_rmse,

            "Train_experiment_IDs": ",".join(map(str, sorted(train_exp_ids))),
            "Val_experiment_IDs": ",".join(map(str, sorted(val_exp_ids)))
        })

        print(
            f"K={K}, Fold={fold_idx}/{K} | "
            f"Train_R2={train_r2:.6f}, Val_R2={val_r2:.6f}, "
            f"Train_RMSE={train_rmse:.6f}, Val_RMSE={val_rmse:.6f}"
        )


XGB Group-level 5-fold cross-validation
K=5, Fold=1/5 | Train_R2=0.998000, Val_R2=0.985252, Train_RMSE=1.594369, Val_RMSE=4.421105
K=5, Fold=2/5 | Train_R2=0.998135, Val_R2=0.979399, Train_RMSE=1.553593, Val_RMSE=5.040981
K=5, Fold=3/5 | Train_R2=0.997880, Val_R2=0.983034, Train_RMSE=1.654472, Val_RMSE=4.597569
K=5, Fold=4/5 | Train_R2=0.998016, Val_R2=0.972734, Train_RMSE=1.588485, Val_RMSE=6.008897
K=5, Fold=5/5 | Train_R2=0.997959, Val_R2=0.980570, Train_RMSE=1.618033, Val_RMSE=4.990164

XGB Group-level 6-fold cross-validation
K=6, Fold=1/6 | Train_R2=0.997724, Val_R2=0.984733, Train_RMSE=1.698809, Val_RMSE=4.544553
K=6, Fold=2/6 | Train_R2=0.998102, Val_R2=0.976970, Train_RMSE=1.569930, Val_RMSE=5.263020
K=6, Fold=3/6 | Train_R2=0.998004, Val_R2=0.982319, Train_RMSE=1.603479, Val_RMSE=4.706916


KeyboardInterrupt: 

## K-fold of DNN

In [ ]:
import pandas as pd
import numpy as np
import os
import random

from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import KFold

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


X_all = data_features.to_numpy(dtype=np.float32)
y_all = data_label.to_numpy(dtype=np.float32).ravel()

print("All data shape:", data.shape)
print("Feature shape:", X_all.shape)
print("Label shape:", y_all.shape)


expected_rows = N_EXPERIMENTS * POINTS_PER_EXPERIMENT
actual_rows = len(data)

print("Expected rows:", expected_rows)
print("Actual rows:", actual_rows)




experiment_ids = np.repeat(
    np.arange(N_EXPERIMENTS),
    POINTS_PER_EXPERIMENT
)

data_with_id = data.copy()
data_with_id["Experiment_ID"] = experiment_ids

all_experiment_ids = np.arange(N_EXPERIMENTS)

K_LIST = [5, 6, 7, 8, 9, 10]

BATCH_SIZE = 256
N_EPOCHS = 50
LR = 0.01

NUM_LAYERS = 3
HIDDEN_SIZE = 12

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class ExcelDataset(Dataset):
    def __init__(self, feature, label):
        self.x = torch.from_numpy(feature).float()
        self.y = torch.from_numpy(label).float()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return {
            "x": self.x[index],
            "y": self.y[index]
        }



class BP(nn.Module):
    def __init__(self, num_layers, hidden_size):
        super(BP, self).__init__()

        self.Liner1 = nn.Linear(10, hidden_size)

        self.queue = [
            nn.Sequential(
                nn.Linear(hidden_size, hidden_size),
                nn.LeakyReLU(0.1)
            )
            for _ in range(num_layers - 1)
        ]

        self.model = nn.Sequential(*self.queue)

        self.Liner2 = nn.Linear(hidden_size, 1)

    def forward(self, input):
        output = self.Liner1(input)
        output = self.model(output)
        output = self.Liner2(output)

        return output



def predict_dnn(model, feature_eval):
    model.eval()

    feature_eval = np.asarray(feature_eval, dtype=np.float32)

    predict_list = []

    with torch.no_grad():
        for test_feature in feature_eval:
            test_feature = torch.from_numpy(test_feature).float()
            test_feature = test_feature.unsqueeze(0)

            predict = model(test_feature)
            predict = predict.squeeze(1)
            predict = predict.squeeze(0)

            predict_list.append(predict.detach().cpu().numpy())

    predict_list = np.array(predict_list).reshape(-1)

    return predict_list



def evaluate_dnn(model, X_eval, y_eval):
    y_pred = predict_dnn(model, X_eval)

    r2 = r2_score(y_eval, y_pred)
    rmse = np.sqrt(mean_squared_error(y_eval, y_pred))

    return r2, rmse



def train_dnn_one_fold(
    feature_train,
    label_train,
    seed=42
):
    set_seed(seed)

    bp = BP(NUM_LAYERS, HIDDEN_SIZE)

    dataset = ExcelDataset(
        feature_train,
        label_train
    )

    g = torch.Generator()
    g.manual_seed(seed)

    dataloader = DataLoader(
        dataset=dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        drop_last=True,
        generator=g
    )

    optimizer = torch.optim.Adam(
        bp.parameters(),
        lr=LR
    )

    MSEloss = nn.MSELoss()

    bp.train()

    for epoch in range(N_EPOCHS):
        epoch_loss_list = []

        for batch in dataloader:
            batch_x = batch["x"]
            batch_y = batch["y"]

            pred = bp(batch_x)
            pred = pred.squeeze(1)

            loss = MSEloss(pred, batch_y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss_list.append(loss.item())

        if (epoch + 1) % 10 == 0:
            print(
                f"Epoch {epoch + 1:03d}/{N_EPOCHS}, "
                f"Batch_loss={np.mean(epoch_loss_list):.6f}"
            )

    return bp

In [2]:
fold_results = []

for K in K_LIST:
    print("\n" + "=" * 70)
    print(f"DNN Group-level {K}-fold cross-validation")
    print("=" * 70)

    kf = KFold(
        n_splits=K,
        shuffle=True,
        random_state=42
    )

    for fold_idx, (train_exp_index, val_exp_index) in enumerate(
        kf.split(all_experiment_ids),
        start=1
    ):
        fold_seed = 1000 + K * 100 + fold_idx

        train_exp_ids = all_experiment_ids[train_exp_index]
        val_exp_ids = all_experiment_ids[val_exp_index]

        train_mask = data_with_id["Experiment_ID"].isin(train_exp_ids).to_numpy()
        val_mask = data_with_id["Experiment_ID"].isin(val_exp_ids).to_numpy()

        feature_train = X_all[train_mask]
        label_train = y_all[train_mask]

        feature_val = X_all[val_mask]
        label_val = y_all[val_mask]

        print(
            f"\nK={K}, Fold={fold_idx}/{K} | "
            f"Train_exp={len(train_exp_ids)}, Val_exp={len(val_exp_ids)} | "
            f"Train_points={len(label_train)}, Val_points={len(label_val)}"
        )

        bp = train_dnn_one_fold(
            feature_train=feature_train,
            label_train=label_train,
            seed=fold_seed
        )

        train_r2, train_rmse = evaluate_dnn(
            bp,
            feature_train,
            label_train
        )

        val_r2, val_rmse = evaluate_dnn(
            bp,
            feature_val,
            label_val
        )

        fold_results.append({
            "Model": "DNN",
            "K": K,
            "Fold": fold_idx,
            "Fold_seed": fold_seed,

            "Train_experiment_number": len(train_exp_ids),
            "Val_experiment_number": len(val_exp_ids),

            "Train_data_points": len(label_train),
            "Val_data_points": len(label_val),

            "Train_R2": train_r2,
            "Val_R2": val_r2,
            "Train_RMSE": train_rmse,
            "Val_RMSE": val_rmse,

            "Train_experiment_IDs": ",".join(map(str, sorted(train_exp_ids))),
            "Val_experiment_IDs": ",".join(map(str, sorted(val_exp_ids)))
        })

        print(
            f"K={K}, Fold={fold_idx}/{K} finished | "
            f"Train_R2={train_r2:.6f}, Val_R2={val_r2:.6f}, "
            f"Train_RMSE={train_rmse:.6f}, Val_RMSE={val_rmse:.6f}"
        )

All data shape: (102172, 11)
Feature shape: (102172, 10)
Label shape: (102172,)
Expected rows: 102172
Actual rows: 102172

DNN Group-level 5-fold cross-validation

K=5, Fold=1/5 | Train_exp=142, Val_exp=36 | Train_points=81508, Val_points=20664
Epoch 010/50, Batch_loss=32.358189
Epoch 020/50, Batch_loss=26.532954


KeyboardInterrupt: 

## K-fold of MQR

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import random

from sklearn.model_selection import KFold
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


data_path = r"C:\Users\24532\Desktop\固废论文二轮修改\周老师意见1\Train_Test_Verification_Data.xlsx"

data = pd.read_excel(data_path,sheet_name="100 divisions_K-folds")


single_bc_path = r"C:\Users\24532\Desktop\固废论文二轮修改\周老师意见1\Train_Test_Verification_Data.xlsx"


data = pd.read_excel(data_path,sheet_name="100 divisions_K-folds")
single_bc_data = pd.read_excel(data_path,sheet_name="1BC")

print("All data shape:", data.shape)
print("All data columns:")
print(data.columns.tolist())

print("\nSingle BC data shape:", single_bc_data.shape)
print("Single BC data columns:")
print(single_bc_data.columns.tolist())


bc_cols = [
    "Cellulose",
    "Hemicellulose",
    "Lignin",
    "PE",
    "PS",
    "PP",
    "PET",
    "PVC",
    "Starch"
]

time_col = "Time"
label_col = "TG"

bc_name_map = {
    "Cellulose": "Cellulose",
    "Hemicellulose": "Hemicellulose",
    "Lignin": "Lignin",
    "PE": "PE",
    "PS": "PS",
    "PP": "PP",
    "PET": "PET",
    "PVC": "PVC",
    "Starch": "Starch"
}


expected_rows = N_EXPERIMENTS * POINTS_PER_EXPERIMENT
actual_rows = len(data)

print("\nExpected rows:", expected_rows)
print("Actual rows:", actual_rows)
print("Actual rows / 574:", actual_rows / POINTS_PER_EXPERIMENT)


missing_cols = [c for c in bc_cols + [time_col, label_col] if c not in data.columns]


experiment_ids = np.repeat(
    np.arange(N_EXPERIMENTS),
    POINTS_PER_EXPERIMENT
)

data_with_id = data.copy()
data_with_id["Experiment_ID"] = experiment_ids

print("\nExperiment_ID count head:")
print(data_with_id["Experiment_ID"].value_counts().sort_index().head())

print("\nExperiment_ID count tail:")
print(data_with_id["Experiment_ID"].value_counts().sort_index().tail())

All data shape: (102172, 11)
All data columns:
['Cellulose', 'Hemicellulose', 'Lignin', 'PE', 'PS', 'PP', 'PET', 'PVC', 'Starch', 'Time', 'TG']

Single BC data shape: (5166, 11)
Single BC data columns:
['Cellulose', 'Hemicellulose', 'Lignin', 'PE', 'PS', 'PP', 'PET', 'PVC', 'Starch', 'Time', 'TG']

Expected rows: 102172
Actual rows: 102172
Actual rows / 574: 178.0

Experiment_ID count head:
Experiment_ID
0    574
1    574
2    574
3    574
4    574
Name: count, dtype: int64

Experiment_ID count tail:
Experiment_ID
173    574
174    574
175    574
176    574
177    574
Name: count, dtype: int64


In [ ]:
def build_single_bc_tg_matrix(single_bc_data, bc_cols, bc_name_map):
    single_tg_dict = {}

    for bc in bc_cols:
        single_col = bc_name_map[bc]

        one_bc_curve = single_bc_data.loc[
            single_bc_data[single_col] == 1,
            ["Time", "TG(wt.%)"]
        ].copy()

        one_bc_curve = one_bc_curve.reset_index(drop=True)
        single_tg_dict[bc] = one_bc_curve["TG(wt.%)"].to_numpy(dtype=np.float32)

    first_bc = bc_cols[0]
    first_single_col = bc_name_map[first_bc]

    time_array = single_bc_data.loc[
        single_bc_data[first_single_col] == 1,
        "Time"
    ].reset_index(drop=True).to_numpy(dtype=np.float32)

    single_tg_df = pd.DataFrame(single_tg_dict)
    single_tg_df.insert(0, time_col, time_array)

    return single_tg_df


single_tg_df = build_single_bc_tg_matrix(
    single_bc_data=single_bc_data,
    bc_cols=bc_cols,
    bc_name_map=bc_name_map
)

print("\nSingle TG matrix shape:", single_tg_df.shape)
display(single_tg_df.head())
display(single_tg_df.tail())



def build_mqr_features_old_fusion(
    df,
    single_tg_df,
    bc_cols,
    time_col,
    use_percent_composition=True
):
    df = df.copy().reset_index(drop=True)

    bcs_values = df[bc_cols].to_numpy(dtype=np.float32)

    max_bcs = np.nanmax(bcs_values)


    data_time = df[time_col].to_numpy(dtype=np.float32)
    single_time = single_tg_df[time_col].to_numpy(dtype=np.float32)

    time_indices = np.array([
        np.argmin(np.abs(single_time - t))
        for t in data_time
    ])

    max_time_diff = np.max(np.abs(data_time - single_time[time_indices]))


    single_tg_values = single_tg_df[bc_cols].to_numpy(dtype=np.float32)
    single_tg_for_each_row = single_tg_values[time_indices, :]

    fused_features = bcs_values * single_tg_for_each_row / 100.0

    fused_df = pd.DataFrame(
        fused_features,
        columns=bc_cols,
        index=df.index
    )

    return fused_df


data_features = build_mqr_features_old_fusion(
    df=data_with_id,
    single_tg_df=single_tg_df,
    bc_cols=bc_cols,
    time_col=time_col,
    use_percent_composition=True
)

data_label = data_with_id[label_col].to_numpy(dtype=np.float32).ravel()

print("\nMQR fused feature shape:", data_features.shape)
print("Label shape:", data_label.shape)
display(data_features.head())


def train_and_evaluate_mqr(X_train, X_val, y_train, y_val):
    poly = PolynomialFeatures(
        degree=2,
        interaction_only=True,
        include_bias=False
    )

    X_train_poly = poly.fit_transform(X_train)
    X_val_poly = poly.transform(X_val)

    X_train_poly[:, 9:] = X_train_poly[:, 9:] / 100.0
    X_val_poly[:, 9:] = X_val_poly[:, 9:] / 100.0

    feature_names = poly.get_feature_names_out(X_train.columns)


    to_keep = [
        idx for idx, name in enumerate(feature_names)
    ]

    X_train_poly_reduced = X_train_poly[:, to_keep]
    X_val_poly_reduced = X_val_poly[:, to_keep]

    reduced_feature_names = [feature_names[idx] for idx in to_keep]

    model = LinearRegression()
    model.fit(X_train_poly_reduced, y_train)

    y_train_pred = model.predict(X_train_poly_reduced)
    y_val_pred = model.predict(X_val_poly_reduced)

    train_r2 = r2_score(y_train, y_train_pred)
    val_r2 = r2_score(y_val, y_val_pred)

    train_mse = mean_squared_error(y_train, y_train_pred)
    val_mse = mean_squared_error(y_val, y_val_pred)

    train_rmse = np.sqrt(train_mse)
    val_rmse = np.sqrt(val_mse)

    coefficients = model.coef_
    intercept = model.intercept_

    coef_df = pd.DataFrame({
        "Feature": reduced_feature_names,
        "Coefficient": coefficients
    })

    return {
        "model": model,
        "poly": poly,
        "to_keep": to_keep,
        "reduced_feature_names": reduced_feature_names,
        "coef_df": coef_df,
        "intercept": intercept,

        "Train_R2": train_r2,
        "Val_R2": val_r2,

        "Train_MSE": train_mse,
        "Val_MSE": val_mse,

        "Train_RMSE": train_rmse,
        "Val_RMSE": val_rmse
    }

In [4]:
K_LIST = [5, 6, 7, 8, 9, 10]

all_experiment_ids = np.arange(N_EXPERIMENTS)

all_results = []

k_summary_list = []


for K in K_LIST:
    print("\n" + "=" * 80)
    print(f"MQR Group-level {K}-fold cross-validation")
    print("=" * 80)

    kf = KFold(
        n_splits=K,
        shuffle=True,
        random_state=42
    )

    fold_train_R2 = np.zeros(K)
    fold_val_R2 = np.zeros(K)

    fold_train_MSE = np.zeros(K)
    fold_val_MSE = np.zeros(K)

    fold_train_RMSE = np.zeros(K)
    fold_val_RMSE = np.zeros(K)

    for fold_idx, (train_exp_index, val_exp_index) in enumerate(
        kf.split(all_experiment_ids),
        start=1
    ):
        train_exp_ids = all_experiment_ids[train_exp_index]
        val_exp_ids = all_experiment_ids[val_exp_index]

        train_mask = data_with_id["Experiment_ID"].isin(train_exp_ids).to_numpy()
        val_mask = data_with_id["Experiment_ID"].isin(val_exp_ids).to_numpy()

        X_train = data_features.loc[train_mask, :].copy()
        X_val = data_features.loc[val_mask, :].copy()

        y_train = data_label[train_mask]
        y_val = data_label[val_mask]

        eval_result = train_and_evaluate_mqr(
            X_train=X_train,
            X_val=X_val,
            y_train=y_train,
            y_val=y_val
        )

        fold_train_R2[fold_idx - 1] = eval_result["Train_R2"]
        fold_val_R2[fold_idx - 1] = eval_result["Val_R2"]

        fold_train_MSE[fold_idx - 1] = eval_result["Train_MSE"]
        fold_val_MSE[fold_idx - 1] = eval_result["Val_MSE"]

        fold_train_RMSE[fold_idx - 1] = eval_result["Train_RMSE"]
        fold_val_RMSE[fold_idx - 1] = eval_result["Val_RMSE"]

        all_results.append({
            "K": K,
            "Fold": fold_idx,

            "Train_experiment_number": len(train_exp_ids),
            "Val_experiment_number": len(val_exp_ids),

            "Train_data_points": len(y_train),
            "Val_data_points": len(y_val),

            "Train_experiment_IDs": ",".join(map(str, sorted(train_exp_ids))),
            "Val_experiment_IDs": ",".join(map(str, sorted(val_exp_ids))),

            "Train_R2": eval_result["Train_R2"],
            "Val_R2": eval_result["Val_R2"],

            "Train_MSE": eval_result["Train_MSE"],
            "Val_MSE": eval_result["Val_MSE"],

            "Train_RMSE": eval_result["Train_RMSE"],
            "Val_RMSE": eval_result["Val_RMSE"],

            "Intercept": eval_result["intercept"]
        })

        print(
            f"K={K}, Fold={fold_idx}/{K} | "
            f"Train_exp={len(train_exp_ids)}, Val_exp={len(val_exp_ids)} | "
            f"Train_R2={eval_result['Train_R2']:.6f}, "
            f"Val_R2={eval_result['Val_R2']:.6f}, "
            f"Train_RMSE={eval_result['Train_RMSE']:.6f}, "
            f"Val_RMSE={eval_result['Val_RMSE']:.6f}"
        )

    k_summary_list.append({
        "K": K,

        "Train_R2_mean": fold_train_R2.mean(),
        "Val_R2_mean": fold_val_R2.mean(),

        "Train_MSE_mean": fold_train_MSE.mean(),
        "Val_MSE_mean": fold_val_MSE.mean(),

        "Train_RMSE_mean": fold_train_RMSE.mean(),
        "Val_RMSE_mean": fold_val_RMSE.mean(),

        "Train_R2_std": fold_train_R2.std(),
        "Val_R2_std": fold_val_R2.std(),

        "Train_RMSE_std": fold_train_RMSE.std(),
        "Val_RMSE_std": fold_val_RMSE.std()
    })

    print(
        f"\nK={K} mean result | "
        f"Train_R2={fold_train_R2.mean():.6f}, "
        f"Val_R2={fold_val_R2.mean():.6f}, "
        f"Train_RMSE={fold_train_RMSE.mean():.6f}, "
        f"Val_RMSE={fold_val_RMSE.mean():.6f}"
    )

All data shape: (102172, 11)
All data columns:
['Cellulose', 'Hemicellulose', 'Lignin', 'PE', 'PS', 'PP', 'PET', 'PVC', 'Starch', 'Time', 'TG']

Single BC data shape: (5166, 11)
Single BC data columns:
['Cellulose', 'Hemicellulose', 'Lignin', 'PE', 'PS', 'PP', 'PET', 'PVC', 'Starch', 'Time', 'TG(wt.%)']

Expected rows: 102172
Actual rows: 102172
Actual rows / 574: 178.0

Experiment_ID count head:
Experiment_ID
0    574
1    574
2    574
3    574
4    574
Name: count, dtype: int64

Experiment_ID count tail:
Experiment_ID
173    574
174    574
175    574
176    574
177    574
Name: count, dtype: int64

Single TG matrix shape: (574, 10)


,Time,Cellulose,Hemicellulose,Lignin,PE,PS,PP,PET,PVC,Starch
0,0.00000,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
1,0.17473,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
2,0.34946,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
3,0.52419,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
4,0.69892,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0


,Time,Cellulose,Hemicellulose,Lignin,PE,PS,PP,PET,PVC,Starch
569,99.421371,5.726536,21.442387,50.137089,-0.695246,0.076772,-0.688827,7.177361,6.370030,6.414624
570,99.596100,5.713284,21.439074,50.127075,-0.698567,0.073435,-0.692159,7.170671,6.366698,6.493578
571,99.770828,5.716607,21.435785,50.127037,-0.698567,0.076773,-0.692176,7.177361,6.363365,6.533001
572,99.945557,5.719922,21.435808,50.093605,-0.691925,0.080111,-0.692210,7.187397,6.356700,6.533001
573,100.120293,5.726552,21.442478,50.056915,-0.685282,0.093463,-0.672318,7.147546,6.320041,6.513236



MQR fused feature shape: (102172, 9)
Label shape: (102172,)


,Cellulose,Hemicellulose,Lignin,PE,PS,PP,PET,PVC,Starch
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



MQR Group-level 5-fold cross-validation
K=5, Fold=1/5 | Train_exp=142, Val_exp=36 | Train_R2=0.957118, Val_R2=0.940987, Train_RMSE=7.383136, Val_RMSE=8.843839
K=5, Fold=2/5 | Train_exp=142, Val_exp=36 | Train_R2=0.954962, Val_R2=-6.009184, Train_RMSE=7.635611, Val_RMSE=92.984032
K=5, Fold=3/5 | Train_exp=142, Val_exp=36 | Train_R2=0.954762, Val_R2=0.948612, Train_RMSE=7.643279, Val_RMSE=8.001435
K=5, Fold=4/5 | Train_exp=143, Val_exp=35 | Train_R2=0.958987, Val_R2=0.863168, Train_RMSE=7.222231, Val_RMSE=13.461057
K=5, Fold=5/5 | Train_exp=143, Val_exp=35 | Train_R2=0.955843, Val_R2=-5.853367, Train_RMSE=7.525127, Val_RMSE=93.720131

K=5 mean result | Train_R2=0.956334, Val_R2=-1.821957, Train_RMSE=7.481877, Val_RMSE=43.402099

MQR Group-level 6-fold cross-validation
K=6, Fold=1/6 | Train_exp=148, Val_exp=30 | Train_R2=0.957061, Val_R2=0.938495, Train_RMSE=7.378157, Val_RMSE=9.121661
K=6, Fold=2/6 | Train_exp=148, Val_exp=30 | Train_R2=0.955937, Val_R2=-0.979914, Train_RMSE=7.563767, V

KeyboardInterrupt: 